In [7]:
import open3d as o3d
import numpy as np
import fpsample  # Gondoskodj róla, hogy telepítve legyen

# Pontfelhő beolvasása
pcd = o3d.io.read_point_cloud("centered.ply")

# Pontok numpy tömbbé konvertálása

pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30))

# 1. Radii lista megadása (ezt gyakran a pontfelhő méretéhez igazítjuk)
radii = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 1.0]

# 2. Ball Pivoting alkalmazása
rec_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    pcd_sampled, o3d.utility.DoubleVector(radii)
)

# 3. Normálok számítása és vizualizálás
rec_mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([rec_mesh],
                                  mesh_show_back_face=True,
                                  window_name="Ball Pivoting rekonstrukció")



In [9]:
import open3d as o3d
import numpy as np
import fpsample  # Gondoskodj róla, hogy telepítve legyen

# Pontfelhő beolvasása
pcd = o3d.io.read_point_cloud("centered.ply")

# Pontok numpy tömbbé konvertálása
vertices = np.asarray(pcd.points)

# FPS minta kiválasztása
sampled_indices = fpsample.bucket_fps_kdtree_sampling(vertices, 20000)
pcd_sampled = pcd.select_by_index(sampled_indices)

# Alpha Shape felület rekonstrukció
alpha = 0.7
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd_sampled, alpha)

# Normálvektorok számítása a hálón
mesh.compute_vertex_normals()

# Simítás (Laplacian smooth)
mesh_smooth = mesh.filter_smooth_laplacian(number_of_iterations=2)
mesh_smooth.compute_vertex_normals()

# Megjelenítés
o3d.visualization.draw_geometries([mesh_smooth],
                                  mesh_show_back_face=True)


In [17]:
import numpy as np
import fpsample
import open3d as o3d  # open3d szükséges a .ply fájl beolvasásához/írásához

# 1. Olvasd be a cave.ply fájlt
pcd = o3d.io.read_point_cloud("cave.ply")
pc_np = np.asarray(pcd.points)  # numpy array-re konvertálás

# 2. Mintavételezés FPS-sel (választhatsz más metódust is)
sampled_idx = fpsample.bucket_fps_kdline_sampling(pc_np, 30000, h=3)  # Ajánlott módszer
sampled_points = pc_np[sampled_idx]

# 3. Új pontfelhő létrehozása a mintavételezett pontokkal
sampled_pcd = o3d.geometry.PointCloud()
sampled_pcd.points = o3d.utility.Vector3dVector(sampled_points)

# 4. Eredmény mentése fájlba
o3d.io.write_point_cloud("cave_sampled.ply", sampled_pcd)

print("Mintavételezés és mentés kész: cave_sampled.ply")


Mintavételezés és mentés kész: cave_sampled.ply


In [ ]:
import open3d as o3d
import time
import numpy as np
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *

class PointCloudViewer:
    def __init__(self, point_cloud_file):
        self.pcd = o3d.io.read_point_cloud(point_cloud_file)
        self.vertices = np.asarray(self.pcd.points)
        self.colors = np.asarray(self.pcd.colors) if self.pcd.has_colors() else np.ones_like(self.vertices) * 0.7

        self.camera_pos = np.array([0.0, 0.0, 0.0], dtype=np.float32)
        self.camera_front = np.array([0.0, 0.0, -1.0], dtype=np.float32)
        self.camera_up = np.array([0.0, 1.0, 0.0], dtype=np.float32)
        self.yaw = -90.0
        self.pitch = 0.0

        self.max_distance = 5.0
        self.fov_cos = np.cos(np.radians(45))
        self.point_size = 2.0
        self.movement_speed = 0.01
        self.mouse_sensitivity = 0.01

        pygame.init()
        self.display = (1280, 720)
        pygame.display.set_mode(self.display, DOUBLEBUF | OPENGL)
        pygame.mouse.set_visible(False)
        pygame.event.set_grab(True)

        gluPerspective(45, (self.display[0] / self.display[1]), 0.1, 100.0)
        glEnable(GL_DEPTH_TEST)
        glPointSize(self.point_size)

        pygame.mouse.get_rel()  # inicializálás: kinullázza az első relatív mozgást

    def update_camera_vectors(self):
        front = np.array([
            np.cos(np.radians(self.yaw)) * np.cos(np.radians(self.pitch)),
            np.sin(np.radians(self.pitch)),
            np.sin(np.radians(self.yaw)) * np.cos(np.radians(self.pitch))
        ], dtype=np.float32)
        self.camera_front = front / np.linalg.norm(front)

    def process_input(self, delta_time):
        running = True
        move_direction = np.zeros(3, dtype=np.float32)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    return False

        keys = pygame.key.get_pressed()
        if keys[pygame.K_w]:
            move_direction += self.camera_front
        if keys[pygame.K_s]:
            move_direction -= self.camera_front
        if keys[pygame.K_a]:
            right = np.cross(self.camera_front, self.camera_up)
            move_direction -= right / np.linalg.norm(right)
        if keys[pygame.K_d]:
            right = np.cross(self.camera_front, self.camera_up)
            move_direction += right / np.linalg.norm(right)
        if keys[pygame.K_SPACE]:
            move_direction += self.camera_up
        if keys[pygame.K_LSHIFT]:
            move_direction -= self.camera_up

        if np.linalg.norm(move_direction) > 0:
            move_direction /= np.linalg.norm(move_direction)
            self.camera_pos += move_direction * self.movement_speed * delta_time

        dx, dy = pygame.mouse.get_rel()
        if dx != 0 or dy != 0:
            self.yaw += dx * self.mouse_sensitivity
            self.pitch -= dy * self.mouse_sensitivity
            self.pitch = max(-89.0, min(89.0, self.pitch))
            self.update_camera_vectors()

        return running

    def get_visible_points(self):
        directions = self.vertices - self.camera_pos
        distances = np.linalg.norm(directions, axis=1)
        directions_normalized = directions / distances[:, np.newaxis]

        dot = np.dot(directions_normalized, self.camera_front)
        mask = (distances < self.max_distance) & (dot > self.fov_cos)

        return self.vertices[mask], self.colors[mask]

    def render(self):
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        cam_target = self.camera_pos + self.camera_front
        gluLookAt(*self.camera_pos, *cam_target, *self.camera_up)

        visible_points, visible_colors = self.get_visible_points()

        glBegin(GL_POINTS)
        for point, color in zip(visible_points, visible_colors):
            glColor3fv(color)
            glVertex3fv(point)
        glEnd()

        pygame.display.flip()


    def run(self):
        clock = pygame.time.Clock()
        self.update_camera_vectors()
        running = True
        last_print_time = time.time()

        while running:
            delta_time = clock.tick(60) / 1000.0
            self.fps = clock.get_fps()
            running = self.process_input(delta_time)
            self.render()

            # Fél másodpercenként írjuk ki a statisztikát
            if time.time() - last_print_time > 0.5:
                visible_points, _ = self.get_visible_points()
                print(f"Látható pontok: {len(visible_points)}, FPS: {int(self.fps)}")
                last_print_time = time.time()

        pygame.mouse.set_visible(True)
        pygame.event.set_grab(False)
        pygame.quit()


if __name__ == "__main__":
    print("""
    Pontfelhő Szimuláció (Fixált vezérlés)

    Vezérlés:
    - W, A, S, D: Mozgás
    - SPACE, LSHIFT: Fel / Le
    - Egér: Kamera forgatás (csak ha tényleg mozog!)
    - ESC: Kilépés
    """)
    viewer = PointCloudViewer("centered.ply")
    viewer.run()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html

    Pontfelhő Szimuláció (Fixált vezérlés)

    Vezérlés:
    - W, A, S, D: Mozgás
    - SPACE, LSHIFT: Fel / Le
    - Egér: Kamera forgatás (csak ha tényleg mozog!)
    - ESC: Kilépés
    
Látható pontok: 19, FPS: 61
Látható pontok: 19, FPS: 50
Látható pontok: 19, FPS: 60
Látható pontok: 19, FPS: 60
Látható pontok: 19, FPS: 58
Látható pontok: 19, FPS: 58
Látható pontok: 19, FPS: 59
Látható pontok: 19, FPS: 59
Látható pontok: 19, FPS: 58
Látható pontok: 19, FPS: 59
Látható pontok: 19, FPS: 58
Látható pontok: 19, FPS: 59
Látható pontok: 19, FPS: 60
